# Aletheia — entraînement sur Colab

Workflow : le code vit sur GitHub (on le modifie en local, on `git push`), les données et les checkpoints vivent sur Google Drive.

**Avant de lancer** : `Exécution > Modifier le type d'exécution > GPU (T4)`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Paramètres

In [ ]:
REPO_URL = "https://github.com/bonlieuguillaume/aletheia.git"
BRANCH = "main"

# Sur ton Google Drive :
DRIVE_ROOT = "/content/drive/MyDrive/aletheia"
DATA_ZIP = f"{DRIVE_ROOT}/data/sar_dataset.zip"   # zip contenant train/ val/ (test/)
RUNS_DIR = f"{DRIVE_ROOT}/runs"                   # checkpoints + logs (survivent à la déconnexion)

LOCAL_DATA = "/content/data"                      # copie locale = I/O bien plus rapide que Drive
CONFIG = "configs/crnn.yaml"                      # ou configs/cnn_baseline.yaml, configs/resnet18.yaml

## 2. Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Récupérer le code (clone ou pull)

Repo privé : ajoute un token GitHub (scope `repo`, lecture seule suffit) dans les **Secrets** Colab (icône 🔑) sous le nom `GITHUB_TOKEN`.

In [ ]:
import os, subprocess

url = REPO_URL
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass  # repo public : pas besoin de token

if os.path.isdir("/content/aletheia/.git"):
    subprocess.run(["git", "-C", "/content/aletheia", "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", "/content/aletheia", "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, url, "/content/aletheia"], check=True)

%cd /content/aletheia
!git log -1 --oneline
!pip install -q -e .

## 4. Données : copie Drive -> disque local de la VM

In [ ]:
if not os.path.isdir(f"{LOCAL_DATA}/train"):
    !mkdir -p {LOCAL_DATA}
    !unzip -q "{DATA_ZIP}" -d {LOCAL_DATA}

from pathlib import Path
for split in ["train", "val", "test"]:
    for cls in ["real", "fake"]:
        d = Path(LOCAL_DATA) / split / cls
        if d.is_dir():
            print(f"{split}/{cls}: {sum(1 for f in d.rglob('*') if f.is_file())} fichiers")

## 5. Entraînement

Si la session coupe : relance les cellules 1→5, l'entraînement reprend depuis `last.pt` sur le Drive.

In [ ]:
!python -m aletheia.train --config {CONFIG} --data-root {LOCAL_DATA} --output-dir {RUNS_DIR}

## 6. Courbes

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

run = Path(RUNS_DIR) / Path(CONFIG).stem
log = pd.read_csv(run / "log.csv")
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(log.epoch, log.train_loss, label="train"); ax[0].plot(log.epoch, log.val_loss, label="val")
ax[0].set_title("loss"); ax[0].legend()
ax[1].plot(log.epoch, log.val_auc, label="AUC"); ax[1].plot(log.epoch, log.val_acc, label="acc")
ax[1].set_title("validation"); ax[1].legend()
plt.show()
log.tail()

## Dev interactif (optionnel)

Pour tester une fonction sans relancer tout l'entraînement.

In [ ]:
%load_ext autoreload
%autoreload 2
import torch
from aletheia.models import build_model
m = build_model({"name": "crnn"}, num_classes=2)
print(m(torch.randn(2, 1, 256, 256)).shape)